# Ollama + Mistral 7B on Colab (GPU)
Run each cell in order. After the last cell, copy the ngrok URL into your local `server/.env`.

In [1]:
# Cell 1 — Verify GPU is available
!nvidia-smi

Tue May 12 09:48:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Cell 2 — Install dependencies, Ollama, and pyngrok
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok -q|
print('Install complete.')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (10.3 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Add

In [10]:
# Cell 3 — Start Ollama server in the background
import subprocess, time, os, shutil, urllib.request

# Refresh PATH so Python finds the ollama binary installed in Cell 2
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
print('Ollama binary:', ollama_bin)

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'

proc = subprocess.Popen(
    [ollama_bin, 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)

# Poll until the server is actually accepting connections (up to 60s)
print('Waiting for Ollama server to be ready...')
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/', timeout=2)
        print(f'Ollama server ready after {i+1}s (PID {proc.pid})')
        break
    except Exception:
        time.sleep(1)
else:
    print('WARNING: Ollama server did not respond within 60s — check for errors')

Ollama binary: /usr/local/bin/ollama
Waiting for Ollama server to be ready...
Ollama server ready after 1s (PID 12120)


In [11]:
# Cell 4 — Pull qwen2.5:14b (~9 GB, fits in T4's 15 GB VRAM)
# This model is significantly better than mistral:7b at structured JSON output.
# Download takes ~5-10 min on Colab.
!ollama pull qwen2.5:14b

In [ ]:
# Cell 5 — Quick smoke test: confirm the model responds
import urllib.request, json

payload = json.dumps({
    'model': 'qwen2.5:14b',
    'messages': [{'role': 'user', 'content': 'Reply with just the word: READY'}],
    'stream': False
}).encode()

req = urllib.request.Request(
    'http://localhost:11434/api/chat',
    data=payload,
    headers={'Content-Type': 'application/json'}
)

# First request loads the model into VRAM — allow up to 3 minutes
print('Sending test prompt (first load may take up to 3 minutes)...')
with urllib.request.urlopen(req, timeout=180) as r:
    result = json.load(r)
print('Model response:', result['message']['content'])
print('Model is working correctly.')

Sending test prompt (first load may take up to 3 minutes)...
Model response:  Ready.
Model is working correctly.


In [14]:
# Cell 6 — Expose Ollama via ngrok
# Get your free token from https://ngrok.com → sign up → Dashboard → Your Authtoken
NGROK_TOKEN = '3BRe0cL8Qf8pB9PvbC9Qhd48A2r_4KyfysVT3EWq6tudYbvGA'  # <-- replace this

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing tunnels first
ngrok.kill()

tunnel = ngrok.connect(11434, 'http')
ollama_url = tunnel.public_url

print('=' * 60)
print('Ollama is live at:', ollama_url)
print('=' * 60)
print()
print('Paste this into your local server/.env:')
print(f'OLLAMA_BASE_URL={ollama_url}')
print('OLLAMA_CONCURRENCY=3')

Ollama is live at: https://unadmonished-maegan-proscientific.ngrok-free.dev

Paste this into your local server/.env:
OLLAMA_BASE_URL=https://unadmonished-maegan-proscientific.ngrok-free.dev
OLLAMA_CONCURRENCY=3


In [15]:
# Cell 7 — Keep-alive: run this to prevent Colab from timing out
# Leave this cell running while you grade. Stop it when done.
import time
print('Keep-alive running. Stop this cell when grading is complete.')
while True:
    time.sleep(30)
    # Ping Ollama every 30 seconds to keep it warm
    try:
        urllib.request.urlopen('http://localhost:11434/', timeout=5)
        print('.', end='', flush=True)
    except:
        print('x', end='', flush=True)

Keep-alive running. Stop this cell when grading is complete.
.....................................................................................

KeyboardInterrupt: 